In [1]:
from pathlib import Path
import os
import db_dtypes

# =========================
# 1. Definir raiz do projeto
# =========================
BASE_DIR = Path.cwd()

# Sobe na árvore até encontrar a pasta "credenciais"
while not (BASE_DIR / "src").exists():
    if BASE_DIR.parent == BASE_DIR:
        raise FileNotFoundError("❌ Pasta 'credenciais' não encontrada em nenhum nível acima.")
    BASE_DIR = BASE_DIR.parent

# =========================
# 2. Montar path do arquivo de credenciais
# =========================
cred_file = "tough-medley-505300-k1-164371097431.json"
cred_path = BASE_DIR / "credenciais" / cred_file

print(f"✅ Arquivo de credenciais localizado em: {cred_path}")

# =========================
# 3. Configurar variável de ambiente para BigQuery
# =========================
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(cred_path)

✅ Arquivo de credenciais localizado em: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json


In [2]:
# =========================
# Parâmetros vindos do Databricks (widgets)
# =========================
import os  # garante que o módulo os está disponível

try:
    # Cria widgets no Databricks
    dbutils.widgets.text("GOOGLE_APPLICATION_CREDENTIALS", "")
    dbutils.widgets.text("BRONZE_CONTAINER", "")
    dbutils.widgets.text("TABLES", "")
    

    # Lê valores dos widgets (passados pelo Data Factory)
    GOOGLE_APPLICATION_CREDENTIALS = dbutils.widgets.get("GOOGLE_APPLICATION_CREDENTIALS")  # caminho do JSON no DBFS
    BRONZE_CONTAINER = dbutils.widgets.get("BRONZE_CONTAINER")  # nome do container no Blob (ex: bronze)
    TABLES = dbutils.widgets.get("TABLES").split(",")  # lista de tabelas recebida do Data Factory

except NameError:
    # Fallback para execução fora do Databricks (ex: testes locais em Jupyter/VSCode)
    print("⚠️ dbutils não encontrado, usando valores locais para teste.")

    GOOGLE_APPLICATION_CREDENTIALS = BASE_DIR / "credenciais" / "tough-medley-505300-k1-164371097431.json"
    BRONZE_CONTAINER = "bronze"

# =========================
# Configura variável de ambiente para BigQuery
# =========================
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(GOOGLE_APPLICATION_CREDENTIALS)


⚠️ dbutils não encontrado, usando valores locais para teste.


In [3]:
# =========================
# Configuração dos clientes
# =========================
from google.cloud import bigquery
from azure.storage.blob import BlobServiceClient

# Cliente BigQuery
try:
    # Se estiver rodando no Databricks, o arquivo deve estar no DBFS
    # Exemplo: /dbfs/FileStore/credentials/tough-medley-505300-k1-164371097431.json
    cred_path = GOOGLE_APPLICATION_CREDENTIALS

    if not os.path.exists(cred_path):
        print(f"⚠️ Arquivo de credenciais não encontrado em {cred_path}.")
    else:
        print(f"✅ Credenciais encontradas em {cred_path}.")

    project_id = os.getenv('GCP_PROJECT_ID', 'tough-medley-505300-k1')
    bq_client = bigquery.Client.from_service_account_json(cred_path, project=project_id)
    print("✅ Cliente BigQuery inicializado com sucesso.")

except Exception as e:
    print(f"❌ Erro ao inicializar cliente BigQuery: {e}")
    bq_client = None  # fallback para evitar crash

# Cliente Azure Blob
try:
    storage_account = os.getenv('AZURE_STORAGE_ACCOUNT', 'meu_storage_account')
    storage_key = os.getenv('AZURE_STORAGE_KEY', 'minha_storage_key')

    blob_service_client = BlobServiceClient(
        f'https://{storage_account}.blob.core.windows.net',
        credential=storage_key
    )
    print("✅ Cliente Azure Blob inicializado com sucesso.")
except Exception as e:
    print(f"❌ Erro ao inicializar cliente Azure Blob: {e}")
    blob_service_client = None


✅ Credenciais encontradas em /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/credenciais/tough-medley-505300-k1-164371097431.json.
✅ Cliente BigQuery inicializado com sucesso.
✅ Cliente Azure Blob inicializado com sucesso.


In [4]:
# =========================
# Função de exportação
# =========================
import datetime as dt

def export_bigquery_table_to_blob(source_table: str, blob_container: str, blob_name: str):
    '''
    Exporta uma tabela do BigQuery direto para Azure Blob em formato Parquet.
    '''
    try:
        query = f'SELECT * FROM `{source_table}`'
        query_job = bq_client.query(query, location='US')

        # Converte resultado para DataFrame Pandas
        df = query_job.to_dataframe()

        # Adiciona colunas de auditoria
        ingested_at = dt.datetime.now(dt.timezone.utc).isoformat()
        df['_ingested_at'] = ingested_at
        df['_source_table'] = source_table

        # Salva localmente em Parquet (no DBFS)
        local_file = BASE_DIR / "dbfs/tmp" / "temp.parquet"
        df.to_parquet(local_file, index=False)

        # Upload para Blob
        blob_client = blob_service_client.get_blob_client(container=blob_container, blob=blob_name)
        with open(local_file, 'rb') as data:
            blob_client.upload_blob(data, overwrite=True)


        print(f'✅ Exportado {source_table} direto para azure://{blob_container}/{blob_name}')

    except Exception as e:
        print(f'❌ Erro ao exportar {source_table} para Blob: {e}')


In [5]:
# Ingestão batch
import datetime as dt

# Gera sufixo AAA-MM-DD
date_suffix = dt.datetime.now().strftime("%Y-%m-%d")

# Monta o nome do blob com sufixo
blob_name_with_date = f"{date_suffix}_uf.parquet"

# Chama a função já com o nome versionado
export_bigquery_table_to_blob(
    source_table="basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    blob_container="bronze",
    blob_name=blob_name_with_date
)



/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf direto para azure://bronze/2026-08-22_uf.parquet


In [6]:
import os
import datetime as dt

# Lista de tabelas
TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio",
    "basedosdados.br_inep_avaliacao_alfabetizacao.municipio"
]

# Data atual para sufixo
date_suffix = dt.datetime.now().strftime("%Y-%m-%d")
container = os.getenv("AZURE_CONTAINER_BRONZE", "bronze")

# Loop sobre todas as tabelas
for table in TABLES:
# Usa apenas o último pedaço do nome da tabela para o arquivo
    table_suffix = table.split(".")[-1]
    blob_name_with_date = f"{date_suffix}_{table_suffix}.parquet"

    export_bigquery_table_to_blob(
        source_table=table,
        blob_container=container,
        blob_name=blob_name_with_date
    )


/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.uf direto para azure://bronze/2026-08-22_uf.parquet


/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil direto para azure://bronze/2026-08-22_meta_alfabetizacao_brasil.parquet


/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf direto para azure://bronze/2026-08-22_meta_alfabetizacao_uf.parquet


/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio direto para azure://bronze/2026-08-22_meta_alfabetizacao_municipio.parquet


/home/linux/miniconda3/envs/teste_2/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Exportado basedosdados.br_inep_avaliacao_alfabetizacao.municipio direto para azure://bronze/2026-08-22_municipio.parquet
